# Setup + Dependencies

In [1]:
# ─── 0. INSTALL DEPENDENCIES ────────────────────────────────────────────────
import subprocess, sys
 
def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
 
pip("ultralytics", "fiftyone", "transformers", "datasets", "peft",
    "evaluate", "rouge_score", "nltk", "faiss-cpu", "tqdm",
    "matplotlib", "seaborn", "pandas", "scikit-learn", "Pillow")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 

# Imports + Setup

In [2]:
# ─── IMPORTS ────────────────────────────────────────────────────────────────
import os, json, random, math, shutil, warnings, re
from pathlib import Path
from tqdm import tqdm
from datetime import datetime
 
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for Kaggle
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
 
warnings.filterwarnings("ignore")
print("✅ Dependencies ready")
print(f"🔥 GPU available: {torch.cuda.is_available()}")

✅ Dependencies ready
🔥 GPU available: True


# Figure Directory + Theme

In [3]:
# ── Figure output directory ──────────────────────────────────────────────────
FIG_DIR = Path("/kaggle/working/figures")
FIG_DIR.mkdir(exist_ok=True)
 
def savefig(name):
    path = FIG_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   📊 Saved figure → {path}")
 
# ── Seaborn theme ────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
PALETTE = {"baseline": "#E07B54", "lora": "#4F9DA6", "accent": "#F2C14E"}

# Phase 1 Start + Dataset Config

In [4]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 1 — DATASET PREPARATION
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 1: DATASET PREPARATION")
print("="*60)
 
import fiftyone as fo
import fiftyone.zoo as foz
 
LABEL_MAP = {
    "Billboard":       "ads",
    "Poster":          "ads",
    "Advertising":     "ads",
    "Plastic bag":     "clutter",
    "Bottle":          "clutter",
    "Waste container": "clutter",
    "Trash":           "clutter",
    "Car":             "vehicle",
    "Bus":             "vehicle",
    "Truck":           "vehicle",
    "Motorcycle":      "vehicle",
    "Van":             "vehicle",
    "Person":          "person",
    "Bench":           "street_furniture",
    "Traffic light":   "street_furniture",
    "Street light":    "street_furniture",
    "Fire hydrant":    "street_furniture",
}
 
CLASSES    = ["ads", "clutter", "vehicle", "person", "street_furniture"]
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}


PHASE 1: DATASET PREPARATION


# Load Dataset

In [5]:
print("📥 Loading Open Images V7 (urban subset)...")
dataset = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=list(LABEL_MAP.keys()),
    max_samples=800,
)
print(f"   Loaded {len(dataset)} samples")

📥 Loading Open Images V7 (urban subset)...
Ignoring invalid classes ['Advertising', 'Trash']
You can view the available classes via `fiftyone.utils.openimages.get_classes()`
 100% |███████████████████| 800/800 [35.8s elapsed, 0s remaining, 20.1 files/s]      
Dataset info written to '/root/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'validation'
Ignoring invalid classes ['Advertising', 'Trash']
You can view the available classes via `fiftyone.utils.openimages.get_classes()`
 100% |█████████████████| 800/800 [9.5s elapsed, 0s remaining, 86.4 samples/s]       
Dataset 'open-images-v7-validation-800' created
   Loaded 800 samples


# Folder Structure


In [6]:
BASE_DIR = Path("/kaggle/working/urban_dataset")
for split in ["train", "val", "test"]:
    (BASE_DIR / "images" / split).mkdir(parents=True, exist_ok=True)
    (BASE_DIR / "labels" / split).mkdir(parents=True, exist_ok=True)

# YOLO Format Conversion Function

In [7]:
def remap_and_export(sample, split):
    img_path = sample.filepath
    dets = sample.ground_truth.detections if sample.ground_truth else []
    yolo_lines = []
    label_counts = {c: 0 for c in CLASSES}
    for det in dets:
        mapped = LABEL_MAP.get(det.label)
        if mapped is None: continue
        idx = CLASS_TO_IDX[mapped]
        x, y, w, h = det.bounding_box
        cx, cy = x + w / 2, y + h / 2
        yolo_lines.append(f"{idx} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
        label_counts[mapped] += 1
    if not yolo_lines:
        return False, {}
    stem = Path(img_path).stem
    shutil.copy(img_path, BASE_DIR / "images" / split / Path(img_path).name)
    (BASE_DIR / "labels" / split / f"{stem}.txt").write_text("\n".join(yolo_lines))
    return True, label_counts

# Train/Val/Test Split

In [8]:
samples = list(dataset)
random.seed(42); random.shuffle(samples)
n = len(samples)
n_train = int(0.70 * n); n_val = int(0.15 * n)
splits_map = (
    [(s, "train") for s in samples[:n_train]] +
    [(s, "val")   for s in samples[n_train:n_train+n_val]] +
    [(s, "test")  for s in samples[n_train+n_val:]]
)

# Export Dataset

In [9]:
exported   = {"train": 0, "val": 0, "test": 0}
split_counts = {"train": {c: 0 for c in CLASSES},
                "val":   {c: 0 for c in CLASSES},
                "test":  {c: 0 for c in CLASSES}}
total_boxes_per_split = {"train": 0, "val": 0, "test": 0}
 
for sample, split in tqdm(splits_map, desc="Exporting YOLO dataset"):
    ok, lc = remap_and_export(sample, split)
    if ok:
        exported[split] += 1
        for cls, cnt in lc.items():
            split_counts[split][cls] += cnt
            total_boxes_per_split[split] += cnt
 
print(f"✅ Exported — train:{exported['train']}  val:{exported['val']}  test:{exported['test']}")

Exporting YOLO dataset: 100%|██████████| 800/800 [00:00<00:00, 1982.60it/s]

✅ Exported — train:560  val:120  test:120


# Create dataset.yaml

In [10]:
yaml_content = f"""
path: {BASE_DIR}
train: images/train
val:   images/val
test:  images/test
nc: {len(CLASSES)}
names: {CLASSES}
"""
(BASE_DIR / "dataset.yaml").write_text(yaml_content)
print("✅ dataset.yaml written")

✅ dataset.yaml written


# Preprocessing Visualization


In [11]:
# ── Preprocessing Visualization ──────────────────────────────────────────────
print("\n📊 Generating preprocessing visualizations...")
 
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Phase 1 — Dataset Preprocessing Summary", fontsize=15, fontweight="bold")


📊 Generating preprocessing visualizations...


Text(0.5, 0.98, 'Phase 1 — Dataset Preprocessing Summary')

# Image Count per Split

In [12]:
# 1a) Split distribution (image counts)
split_labels = list(exported.keys())
split_vals   = list(exported.values())
colors_split = [PALETTE["baseline"], PALETTE["lora"], PALETTE["accent"]]
bars = axes[0].bar(split_labels, split_vals, color=colors_split, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, split_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 str(val), ha="center", va="bottom", fontweight="bold")
axes[0].set_title("Image Count per Split", fontweight="bold")
axes[0].set_ylabel("Number of Images")
axes[0].set_ylim(0, max(split_vals) * 1.2)

(0.0, 672.0)

# Class Distribution (Stacked Bar)

In [13]:
# 1b) Class distribution across all splits (stacked bar)
all_class_counts = {c: sum(split_counts[sp][c] for sp in ["train","val","test"]) for c in CLASSES}
cls_colors = sns.color_palette("Set2", len(CLASSES))
xs = np.arange(len(CLASSES))
bottom = np.zeros(len(CLASSES))
for si, split in enumerate(["train", "val", "test"]):
    vals = [split_counts[split][c] for c in CLASSES]
    axes[1].bar(xs, vals, bottom=bottom, label=split.capitalize(),
                color=colors_split[si], alpha=0.85, edgecolor="white")
    bottom += np.array(vals)
axes[1].set_xticks(xs); axes[1].set_xticklabels(CLASSES, rotation=15, ha="right")
axes[1].set_title("Label Distribution per Class (stacked splits)", fontweight="bold")
axes[1].set_ylabel("Annotation Count"); axes[1].legend()

In [14]:
# 1c) Boxes per split (pie)
axes[2].pie(
    [total_boxes_per_split[s] for s in ["train","val","test"]],
    labels=[f"{s.capitalize()}\n({total_boxes_per_split[s]})" for s in ["train","val","test"]],
    colors=colors_split, autopct="%1.1f%%", startangle=140,
    wedgeprops=dict(edgecolor="white", linewidth=1.5)
)
axes[2].set_title("Bounding Box Share per Split", fontweight="bold")
 
savefig("01_preprocessing.png")

   📊 Saved figure → /kaggle/working/figures/01_preprocessing.png


In [15]:
# ── Per-class image count grid ───────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4))
fig2.suptitle("Phase 1 — Per-class Annotation Counts (all splits)", fontsize=13, fontweight="bold")
df_cls = pd.DataFrame(split_counts).T  # rows=splits, cols=classes
df_cls.plot(kind="bar", ax=ax2, color=cls_colors, edgecolor="white", linewidth=1.2)
ax2.set_xticklabels(["Train", "Val", "Test"], rotation=0)
ax2.set_ylabel("Annotation Count"); ax2.legend(title="Class", bbox_to_anchor=(1.01, 1))
savefig("01b_class_per_split.png")

   📊 Saved figure → /kaggle/working/figures/01b_class_per_split.png


In [16]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 2 — YOLO BASELINE TRAINING
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 2: YOLO TRAINING")
print("="*60)
 
from ultralytics import YOLO


PHASE 2: YOLO TRAINING
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [17]:
yolo_model = YOLO("yolov8s.pt")
results = yolo_model.train(
    data=str(BASE_DIR / "dataset.yaml"),
    epochs=30, imgsz=640, batch=16,
    device=0 if torch.cuda.is_available() else "cpu",
    project="/kaggle/working/yolo_runs", name="urban_v1",
    exist_ok=True, verbose=False,
)
print("✅ YOLO training complete")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/urban_dataset/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=urban_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, p

In [18]:
val_results = yolo_model.val(data=str(BASE_DIR / "dataset.yaml"), split="test")
print(f"   mAP@50    : {val_results.box.map50:.4f}")
print(f"   Precision : {val_results.box.mp:.4f}")
print(f"   Recall    : {val_results.box.mr:.4f}")

Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2536.3±892.2 MB/s, size: 258.2 KB)
val: Scanning /kaggle/working/urban_dataset/labels/test... 120 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 120/120 1.3Kit/s 0.1s
val: New cache created: /kaggle/working/urban_dataset/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 3.2it/s 2.5s0.2ss
                   all        120        346      0.468      0.341      0.339      0.256
                   ads          5         35      0.444     0.0254     0.0791     0.0404
               clutter          3          4      0.522       0.25      0.328      0.287
               vehicle         52         91      0.623      0.763      0.773      0.618
                person         69        216      0.284

In [19]:
BEST_YOLO = "/kaggle/working/yolo_runs/urban_v1/weights/best.pt"
yolo_model = YOLO(BEST_YOLO)
print(f"✅ Best YOLO model loaded")

✅ Best YOLO model loaded


In [20]:
# ── YOLO training curve visualization ───────────────────────────────────────
results_csv = Path("/kaggle/working/yolo_runs/urban_v1/results.csv")
if results_csv.exists():
    df_yolo = pd.read_csv(results_csv)
    df_yolo.columns = df_yolo.columns.str.strip()
 
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Phase 2 — YOLO Training Curves", fontsize=15, fontweight="bold")
 
    metrics_to_plot = [
        ("train/box_loss",   "Train Box Loss",   PALETTE["baseline"]),
        ("train/cls_loss",   "Train Cls Loss",   PALETTE["lora"]),
        ("train/dfl_loss",   "Train DFL Loss",   PALETTE["accent"]),
        ("metrics/mAP50(B)", "Val mAP@50",       "#7B68EE"),
        ("metrics/precision(B)", "Val Precision",  "#20B2AA"),
        ("metrics/recall(B)",    "Val Recall",     "#FF6B6B"),
    ]
    for ax, (col, title, color) in zip(axes.flatten(), metrics_to_plot):
        if col in df_yolo.columns:
            ax.plot(df_yolo["epoch"], df_yolo[col], color=color, linewidth=2.2, marker="o",
                    markersize=3, markevery=5)
            ax.set_title(title, fontweight="bold"); ax.set_xlabel("Epoch"); ax.grid(alpha=0.4)
        else:
            ax.set_visible(False)
    savefig("02_yolo_training_curves.png")

   📊 Saved figure → /kaggle/working/figures/02_yolo_training_curves.png


In [21]:
# ── Per-class YOLO bar chart ─────────────────────────────────────────────────
per_class_map  = {
    "ads":             val_results.box.maps[0] if len(val_results.box.maps) > 0 else 0,
    "clutter":         val_results.box.maps[1] if len(val_results.box.maps) > 1 else 0,
    "vehicle":         val_results.box.maps[2] if len(val_results.box.maps) > 2 else 0,
    "person":          val_results.box.maps[3] if len(val_results.box.maps) > 3 else 0,
    "street_furniture":val_results.box.maps[4] if len(val_results.box.maps) > 4 else 0,
}
fig, ax = plt.subplots(figsize=(9, 5))
fig.suptitle("Phase 2 — YOLO Per-class mAP@50:95 on Test Set", fontsize=13, fontweight="bold")
bar_colors = sns.color_palette("Set2", len(CLASSES))
bars = ax.barh(list(per_class_map.keys()), list(per_class_map.values()),
               color=bar_colors, edgecolor="white")
for bar, val in zip(bars, per_class_map.values()):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontweight="bold")
ax.set_xlabel("mAP@50:95"); ax.set_xlim(0, 1.0)
savefig("02b_yolo_per_class.png")

   📊 Saved figure → /kaggle/working/figures/02b_yolo_per_class.png


# Livability Scoring


In [23]:
print("\n" + "="*60)
print("PHASE 3: LIVABILITY SCORING")
print("="*60)


PHASE 3: LIVABILITY SCORING


In [24]:
SCORE_WEIGHTS = {"ads": 2.0, "clutter": 3.0, "vehicle": 2.0,
                 "person": 1.5, "street_furniture": 0.5}

In [25]:
def compute_score(counts: dict) -> float:
    penalty = sum(SCORE_WEIGHTS.get(cls, 0) * cnt for cls, cnt in counts.items())
    return round(max(0.0, min(100.0, 100 - penalty)), 2)

In [26]:
def run_yolo_on_image(img_path: str) -> dict:
    result = yolo_model(img_path, verbose=False)[0]
    counts = {c: 0 for c in CLASSES}
    for box in result.boxes:
        counts[CLASSES[int(box.cls)]] += 1
    return counts

In [27]:
test_imgs = list((BASE_DIR / "images" / "test").glob("*.jpg"))
if test_imgs:
    sample_img = str(test_imgs[0])
    counts = run_yolo_on_image(sample_img)
    score  = compute_score(counts)
    print(f"   Sample: counts={counts}  score={score}")
print("✅ Scoring logic verified")

   Sample: counts={'ads': 0, 'clutter': 0, 'vehicle': 0, 'person': 10, 'street_furniture': 0}  score=85.0
✅ Scoring logic verified


# LLM Dataset Generation

In [28]:

print("\n" + "="*60)
print("PHASE 4: LLM DATASET GENERATION (dual-prompt)")
print("="*60)


PHASE 4: LLM DATASET GENERATION (dual-prompt)


In [29]:
# ─────────────────────────────────────────────────────────────────────────
# BASELINE PROMPT STYLE — simple, terse, raw key=value
# ─────────────────────────────────────────────────────────────────────────
def build_baseline_input(counts, score):
    """Minimalist prompt — no context, no task framing."""
    parts = " | ".join(f"{k}:{v}" for k, v in counts.items())
    return f"data: {parts} | score:{score}"

In [30]:
# ─────────────────────────────────────────────────────────────────────────
# LoRA PROMPT STYLE — structured, detailed, instruction-based
# ─────────────────────────────────────────────────────────────────────────
def build_lora_input(counts, score):
    """Structured prompt with explicit instruction and role."""
    parts = ", ".join(f"{k}={v}" for k, v in counts.items())
    return (
        f"Task: Generate a 2-sentence urban livability report.\n"
        f"Detections: {parts}\n"
        f"Livability Score: {score}/100\n"
        f"Output a precise assessment:"
    )

In [31]:
def _score_band(s):
    return "good" if s >= 70 else ("moderate" if s >= 40 else "poor")
 
def _severity(n):
    return "no" if n == 0 else ("minimal" if n <= 1 else ("moderate" if n <= 3 else "heavy"))

In [32]:
# ── Diverse baseline targets ──
BASELINE_SENTENCE1 = {
    "good": [
        "Street conditions appear favourable, scoring {score} out of 100.",
        "This location achieved a livability rating of {score}, reflecting healthy urban quality.",
        "With a score of {score}, the street environment is generally pleasant.",
    ],
    "moderate": [
        "The area received a livability score of {score}, suggesting room for improvement.",
        "Urban conditions here are mixed, yielding a score of {score} on our scale.",
        "A score of {score} indicates moderate quality in this street environment.",
    ],
    "poor": [
        "At {score} out of 100, this location shows significant livability concerns.",
        "The street environment is strained, registering a score of just {score}.",
        "Poor urban conditions drive the livability index down to {score}.",
    ],
}

In [33]:
BASELINE_SENTENCE2_TEMPLATES = [
    "Waste accumulation ({clutter} items) combined with {vehicle} vehicles are the primary detractors.",
    "The {clutter} clutter objects and {ads} ad displays significantly worsen street quality.",
    "Traffic density stands at {vehicle} vehicles, causing congestion alongside {person} pedestrians.",
    "Heavy vehicular presence ({vehicle} units) with {ads} billboards increases visual and physical load.",
    "Advertising clutter ({ads} boards) and waste ({clutter} items) create a visually noisy street.",
    "{ads} advertisements compete for attention in a space already busy with {vehicle} vehicles.",
    "Crowded footpaths ({person} pedestrians) and {vehicle} vehicles indicate high-density usage.",
    "Foot traffic ({person} people) paired with {clutter} waste objects strain the environment.",
    "With minimal clutter ({clutter} items) and {person} pedestrians, the street is relatively clean.",
    "Balanced urban activity: {vehicle} vehicles, {person} pedestrians, and {ads} ad displays.",
]

In [34]:
def _pick_baseline_s2_index(counts):
    if counts["clutter"] >= 4: return random.choice([0, 1])
    if counts["vehicle"] >= 6: return random.choice([2, 3])
    if counts["ads"] >= 4:     return random.choice([4, 5])
    if counts["person"] >= 7:  return random.choice([6, 7])
    return random.choice([8, 9])
 
def generate_baseline_target(counts, score):
    band = _score_band(score)
    s1_template = random.choice(BASELINE_SENTENCE1[band])
    s1 = s1_template.format(score=score)
    idx = _pick_baseline_s2_index(counts)
    s2 = BASELINE_SENTENCE2_TEMPLATES[idx].format(**counts)
    return f"{s1} {s2}"

In [35]:
LORA_S1_TEMPLATES = {
    "good":     "The urban area has a livability score of {score}, indicating good street conditions.",
    "moderate": "The urban area has a livability score of {score}, indicating moderate street conditions.",
    "poor":     "The urban area has a livability score of {score}, indicating poor street conditions.",
}
 
LORA_S2_TEMPLATES = {
    "clutter_heavy":  "Heavy waste presence ({clutter} items) and {ads} advertisements severely reduce environmental quality.",
    "clutter_mod":    "Moderate clutter ({clutter} items) alongside {vehicle} vehicles contributes to reduced livability.",
    "ads_heavy":      "Visual pollution from {ads} advertisements and {vehicle} vehicles lowers the livability score.",
    "vehicle_heavy":  "High vehicular density ({vehicle} vehicles) causes congestion, affecting {person} pedestrians negatively.",
    "person_heavy":   "Dense foot traffic ({person} people) and {vehicle} vehicles create a congested urban environment.",
    "clean":          "Minimal clutter ({clutter} items) and {person} pedestrians reflect a relatively clean street environment.",
    "balanced":       "With {vehicle} vehicles, {person} pedestrians, and {ads} ads, the street shows balanced urban activity.",
}

In [37]:
def _pick_lora_s2_key(counts):
    if counts["clutter"] >= 4:   return "clutter_heavy"
    if counts["clutter"] >= 2:   return "clutter_mod"
    if counts["ads"] >= 4:       return "ads_heavy"
    if counts["vehicle"] >= 6:   return "vehicle_heavy"
    if counts["person"] >= 7:    return "person_heavy"
    if counts["clutter"] == 0 and counts["ads"] <= 1: return "clean"
    return "balanced"
 
def generate_lora_target(counts, score):
    band = _score_band(score)
    s1 = LORA_S1_TEMPLATES[band].format(score=score)
    s2 = LORA_S2_TEMPLATES[_pick_lora_s2_key(counts)].format(**counts)
    return f"{s1} {s2}"

In [38]:
# ── Generate dataset (shared counts, split targets) ──────────────────────
random.seed(42)
llm_data = []       # used for T5 baseline
llm_lora_data = []  # used for LoRA

In [39]:
# Real YOLO predictions
for img_path in tqdm(test_imgs[:200], desc="Real predictions"):
    counts = run_yolo_on_image(str(img_path))
    score  = compute_score(counts)
    llm_data.append({
        "input_text":  build_baseline_input(counts, score),
        "target_text": generate_baseline_target(counts, score),
    })
    llm_lora_data.append({
        "input_text":  build_lora_input(counts, score),
        "target_text": generate_lora_target(counts, score),
    })

Real predictions: 100%|██████████| 120/120 [00:02<00:00, 43.35it/s]


In [40]:
# Synthetic augmentation
for _ in range(2800):
    counts = {
        "ads":             random.randint(0, 8),
        "clutter":         random.randint(0, 6),
        "vehicle":         random.randint(0, 10),
        "person":          random.randint(0, 12),
        "street_furniture":random.randint(0, 4),
    }
    score = compute_score(counts)
    llm_data.append({
        "input_text":  build_baseline_input(counts, score),
        "target_text": generate_baseline_target(counts, score),
    })
    llm_lora_data.append({
        "input_text":  build_lora_input(counts, score),
        "target_text": generate_lora_target(counts, score),
    })

In [41]:
# Shuffle with same seed for aligned split
combined = list(zip(llm_data, llm_lora_data))
random.shuffle(combined)
llm_data, llm_lora_data = zip(*combined)
llm_data, llm_lora_data = list(llm_data), list(llm_lora_data)

In [42]:
print(f"✅ Baseline dataset : {len(llm_data)} samples")
print(f"✅ LoRA dataset     : {len(llm_lora_data)} samples")
print(f"\n   Baseline input  : {llm_data[0]['input_text']}")
print(f"   Baseline target : {llm_data[0]['target_text']}")
print(f"\n   LoRA input      : {llm_lora_data[0]['input_text']}")
print(f"   LoRA target     : {llm_lora_data[0]['target_text']}")

✅ Baseline dataset : 2920 samples
✅ LoRA dataset     : 2920 samples

   Baseline input  : data: ads:7 | clutter:4 | vehicle:0 | person:1 | street_furniture:0 | score:72.5
   Baseline target : This location achieved a livability rating of 72.5, reflecting healthy urban quality. Waste accumulation (4 items) combined with 0 vehicles are the primary detractors.

   LoRA input      : Task: Generate a 2-sentence urban livability report.
Detections: ads=7, clutter=4, vehicle=0, person=1, street_furniture=0
Livability Score: 72.5/100
Output a precise assessment:
   LoRA target     : The urban area has a livability score of 72.5, indicating good street conditions. Heavy waste presence (4 items) and 7 advertisements severely reduce environmental quality.


In [43]:
# ── Dataset statistics visualization ─────────────────────────────────────
scores_all = []
for item in llm_data:
    try:
        s = float(item["input_text"].split("score:")[-1].strip())
        scores_all.append(s)
    except:
        pass

In [44]:
target_lengths_baseline = [len(d["target_text"].split()) for d in llm_data]
target_lengths_lora     = [len(d["target_text"].split()) for d in llm_lora_data]

In [45]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Phase 4 — LLM Dataset Statistics", fontsize=14, fontweight="bold")
 
# Score distribution
if scores_all:
    axes[0].hist(scores_all, bins=30, color=PALETTE["accent"], edgecolor="white", linewidth=0.8)
    axes[0].set_title("Score Distribution", fontweight="bold")
    axes[0].set_xlabel("Livability Score"); axes[0].set_ylabel("Count")
 
# Target length comparison
axes[1].hist(target_lengths_baseline, bins=20, alpha=0.7, color=PALETTE["baseline"],
             label="Baseline (diverse)", edgecolor="white")
axes[1].hist(target_lengths_lora, bins=20, alpha=0.7, color=PALETTE["lora"],
             label="LoRA (templated)", edgecolor="white")
axes[1].set_title("Target Sentence Length Distribution", fontweight="bold")
axes[1].set_xlabel("Word Count"); axes[1].legend()
 
# Prompt length comparison
axes[2].bar(["Baseline\n(terse)","LoRA\n(structured)"],
            [np.mean([len(d["input_text"]) for d in llm_data]),
             np.mean([len(d["input_text"]) for d in llm_lora_data])],
            color=[PALETTE["baseline"], PALETTE["lora"]], edgecolor="white")
axes[2].set_title("Avg Prompt Length (chars)", fontweight="bold")
axes[2].set_ylabel("Characters")
 
savefig("04_dataset_stats.png")

   📊 Saved figure → /kaggle/working/figures/04_dataset_stats.png


In [46]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 5 — BASELINE T5 TRAINING (with diverse prompts)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 5: BASELINE T5 TRAINING")
print("="*60)


PHASE 5: BASELINE T5 TRAINING


In [47]:
from transformers import (
    T5ForConditionalGeneration, T5Tokenizer,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import evaluate
 
MODEL_NAME = "t5-small"
tokenizer  = T5Tokenizer.from_pretrained(MODEL_NAME)
 
MAX_INPUT  = 96
MAX_TARGET = 96

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [48]:
def tokenize(batch):
    model_inputs = tokenizer(batch["input_text"],
                             max_length=MAX_INPUT, truncation=True, padding="max_length")
    labels = tokenizer(batch["target_text"],
                       max_length=MAX_TARGET, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [49]:
split_idx   = int(0.85 * len(llm_data))
train_ds    = Dataset.from_list(llm_data[:split_idx]).map(tokenize, batched=True,
              remove_columns=["input_text","target_text"])
eval_ds     = Dataset.from_list(llm_data[split_idx:]).map(tokenize, batched=True,
              remove_columns=["input_text","target_text"])

Map:   0%|          | 0/2482 [00:00<?, ? examples/s]

Map:   0%|          | 0/438 [00:00<?, ? examples/s]

In [50]:
rouge_metric = evaluate.load("rouge")

In [51]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

In [52]:
t5_base = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
collator = DataCollatorForSeq2Seq(tokenizer, model=t5_base, padding=True)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [53]:
train_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/t5_baseline",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [54]:
baseline_epoch_metrics = {"rouge1":[], "rouge2":[], "rougeL":[], "train_loss":[]}
 
from transformers import TrainerCallback
 
class EpochMetricsCallback(TrainerCallback):
    def __init__(self, store): self.store = store
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics:
            self.store["rouge1"].append(metrics.get("eval_rouge1", 0))
            self.store["rouge2"].append(metrics.get("eval_rouge2", 0))
            self.store["rougeL"].append(metrics.get("eval_rougeL", 0))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs and "loss" in logs:
            self.store["train_loss"].append(logs.get("loss", 0))
 
baseline_cb = EpochMetricsCallback(baseline_epoch_metrics)

In [55]:
trainer_base = Seq2SeqTrainer(
    model=t5_base, args=train_args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[baseline_cb],
)
trainer_base.train()
print("✅ T5 baseline training complete")

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.084650,0.046291,0.364400,0.248900,0.337300,0.338100
2,0.041117,0.027405,0.492700,0.372500,0.458300,0.457800
3,0.030738,0.022040,0.507000,0.392900,0.475900,0.476000
4,0.026539,0.020766,0.477000,0.361900,0.444900,0.445200
5,0.026306,0.020137,0.502900,0.389200,0.470200,0.470400
6,0.024036,0.020085,0.468900,0.357800,0.438900,0.439300
7,0.023636,0.019648,0.466300,0.364000,0.440600,0.440900
8,0.022741,0.019331,0.471100,0.370100,0.445600,0.445700


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


✅ T5 baseline training complete


In [56]:
def generate_text(model, text, max_new=96, num_beams=4):
    enc = tokenizer(text, return_tensors="pt", max_length=MAX_INPUT, truncation=True)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, num_beams=num_beams, early_stopping=True)
    return tokenizer.decode(out[0], skip_special_tokens=True)

In [57]:
import nltk
nltk.download("punkt", quiet=True); nltk.download("punkt_tab", quiet=True)
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
bleu_smooth = SmoothingFunction().method1

In [58]:
def full_evaluate(model, dataset_list, label):
    model.eval()
    preds, refs = [], []
    for item in tqdm(dataset_list, desc=f"Evaluating {label}"):
        pred = generate_text(model, item["input_text"])
        preds.append(pred); refs.append(item["target_text"])
    rouge_scores = rouge_metric.compute(predictions=preds, references=refs)
    bleu_scores  = [
        sentence_bleu([r.split()], p.split(), smoothing_function=bleu_smooth,
                      weights=(0.5, 0.5, 0, 0))
        for p, r in zip(preds, refs)
    ]
    return {
        "ROUGE-1": round(rouge_scores["rouge1"], 4),
        "ROUGE-2": round(rouge_scores["rouge2"], 4),
        "ROUGE-L": round(rouge_scores["rougeL"], 4),
        "BLEU":    round(float(np.mean(bleu_scores)), 4),
    }, preds

In [59]:
eval_subset_base = llm_data[split_idx: split_idx + 200]
baseline_metrics, baseline_preds = full_evaluate(t5_base, eval_subset_base, "Baseline T5")
 
print(f"\n📊 Baseline T5 metrics: {baseline_metrics}")

Evaluating Baseline T5: 100%|██████████| 200/200 [02:07<00:00,  1.57it/s]



📊 Baseline T5 metrics: {'ROUGE-1': np.float64(0.5954), 'ROUGE-2': np.float64(0.4864), 'ROUGE-L': np.float64(0.5561), 'BLEU': 0.4584}


In [60]:
# ── Baseline training visualization ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phase 5 — Baseline T5 Training", fontsize=14, fontweight="bold")
 
if baseline_epoch_metrics["rouge1"]:
    epochs_r = range(1, len(baseline_epoch_metrics["rouge1"]) + 1)
    axes[0].plot(epochs_r, baseline_epoch_metrics["rouge1"], marker="o", color=PALETTE["baseline"],
                 label="ROUGE-1", linewidth=2)
    axes[0].plot(epochs_r, baseline_epoch_metrics["rouge2"], marker="s", color=PALETTE["lora"],
                 label="ROUGE-2", linewidth=2)
    axes[0].plot(epochs_r, baseline_epoch_metrics["rougeL"], marker="^", color=PALETTE["accent"],
                 label="ROUGE-L", linewidth=2)
    axes[0].set_title("Eval ROUGE per Epoch", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("ROUGE Score")
    axes[0].legend(); axes[0].set_ylim(0, 1.05)
 
if baseline_epoch_metrics["train_loss"]:
    steps = range(1, len(baseline_epoch_metrics["train_loss"]) + 1)
    axes[1].plot(steps, baseline_epoch_metrics["train_loss"], color="#E07B54", linewidth=1.8, alpha=0.8)
    axes[1].set_title("Training Loss", fontweight="bold")
    axes[1].set_xlabel("Logging Step"); axes[1].set_ylabel("Loss")
 
savefig("05_baseline_training.png")

   📊 Saved figure → /kaggle/working/figures/05_baseline_training.png


# LoRA FINE-TUNING

In [61]:
# ═══════════════════════════════════════════════════════════════════════════
# PHASE 6 — LoRA FINE-TUNING (structured prompts)
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("PHASE 6: LoRA FINE-TUNING")
print("="*60)


PHASE 6: LoRA FINE-TUNING


In [62]:
from peft import get_peft_model, LoraConfig, TaskType
 
split_idx_lora = int(0.85 * len(llm_lora_data))
train_ds_lora  = Dataset.from_list(llm_lora_data[:split_idx_lora]).map(
    tokenize, batched=True, remove_columns=["input_text","target_text"])
eval_ds_lora   = Dataset.from_list(llm_lora_data[split_idx_lora:]).map(
    tokenize, batched=True, remove_columns=["input_text","target_text"])

Map:   0%|          | 0/2482 [00:00<?, ? examples/s]

Map:   0%|          | 0/438 [00:00<?, ? examples/s]

In [63]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=64,
    target_modules=["q", "v", "k", "o"],
    lora_dropout=0.05, bias="none",
)

In [64]:
t5_lora_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)
t5_lora_model = get_peft_model(t5_lora_model, lora_config)
t5_lora_model.print_trainable_parameters()

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

trainable params: 1,179,648 || all params: 61,686,272 || trainable%: 1.9123


In [65]:
lora_epoch_metrics = {"rouge1":[], "rouge2":[], "rougeL":[], "train_loss":[]}
lora_cb = EpochMetricsCallback(lora_epoch_metrics)

In [66]:
lora_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/t5_lora",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=3e-4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [67]:
collator_lora = DataCollatorForSeq2Seq(tokenizer, model=t5_lora_model, padding=True)
 
trainer_lora = Seq2SeqTrainer(
    model=t5_lora_model, args=lora_args,
    train_dataset=train_ds_lora, eval_dataset=eval_ds_lora,
    processing_class=tokenizer, data_collator=collator_lora,
    compute_metrics=compute_metrics,
    callbacks=[lora_cb],
)
trainer_lora.train()
print("✅ LoRA fine-tuning complete")

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,0.862827,0.240022,0.689300,0.649300,0.684400,0.684600
2,0.112435,0.051300,0.689300,0.649300,0.684400,0.684600
3,0.059569,0.025806,0.699400,0.678100,0.698200,0.698300
4,0.040887,0.014377,0.698000,0.674200,0.696300,0.696400
5,0.029394,0.009713,0.700700,0.683000,0.700500,0.700600
6,0.021313,0.006774,0.700700,0.683000,0.700500,0.700600
7,0.016822,0.006902,0.700700,0.683000,0.700500,0.700600
8,0.015596,0.006137,0.700600,0.683000,0.700500,0.700600
9,0.014073,0.005527,0.700900,0.683700,0.700900,0.700900
10,0.013768,0.005558,0.700700,0.683200,0.700600,0.700700


✅ LoRA fine-tuning complete


In [ ]:
eval_subset_lora = llm_lora_data[split_idx_lora: split_idx_lora + 200]
lora_metrics, lora_preds = full_evaluate(t5_lora_model, eval_subset_lora, "LoRA T5")
 
print(f"\n📊 LoRA T5 metrics: {lora_metrics}")

Evaluating LoRA T5:  34%|███▎      | 67/200 [01:04<02:04,  1.07it/s]

# LoRA Training Visualization


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Phase 6 — LoRA Fine-Tuning Training Curves", fontsize=14, fontweight="bold")
 
if lora_epoch_metrics["rouge1"]:
    epochs_l = range(1, len(lora_epoch_metrics["rouge1"]) + 1)
    axes[0].plot(epochs_l, lora_epoch_metrics["rouge1"], marker="o", color=PALETTE["lora"],
                 label="ROUGE-1", linewidth=2)
    axes[0].plot(epochs_l, lora_epoch_metrics["rouge2"], marker="s", color="#7B68EE",
                 label="ROUGE-2", linewidth=2)
    axes[0].plot(epochs_l, lora_epoch_metrics["rougeL"], marker="^", color=PALETTE["accent"],
                 label="ROUGE-L", linewidth=2)
    axes[0].set_title("Eval ROUGE per Epoch", fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("ROUGE Score")
    axes[0].legend(); axes[0].set_ylim(0, 1.05)
 
if lora_epoch_metrics["train_loss"]:
    steps = range(1, len(lora_epoch_metrics["train_loss"]) + 1)
    axes[1].plot(steps, lora_epoch_metrics["train_loss"], color=PALETTE["lora"],
                 linewidth=1.8, alpha=0.85)
    axes[1].set_title("Training Loss", fontweight="bold")
    axes[1].set_xlabel("Logging Step"); axes[1].set_ylabel("Loss")
 
savefig("06_lora_training.png")

# Metric comparison

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# 📊 BASELINE vs LoRA — METRICS COMPARISON
# ═══════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("📊 BASELINE vs LoRA — METRICS COMPARISON")
print("="*60)
 
metrics_order = ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BLEU"]
header = f"{'Metric':<12} {'Baseline':>12} {'LoRA':>12} {'Δ':>16} {'Winner':>10}"
print(header); print("-"*len(header))
 
comparison_results = {}
for m in metrics_order:
    bv, lv = baseline_metrics[m], lora_metrics[m]
    delta = lv - bv
    pct   = (delta / bv * 100) if bv > 0 else 0
    winner = "LoRA ✅" if lv >= bv else "Base ✅"
    print(f"{m:<12} {bv:>12.4f} {lv:>12.4f} {delta:>+10.4f} ({pct:>+5.1f}%) {winner:>8}")
    comparison_results[m] = {"baseline": bv, "lora": lv, "delta": delta, "pct": pct}
print("-"*len(header))
 
lora_wins = sum(1 for m in metrics_order if lora_metrics[m] >= baseline_metrics[m])
print(f"\n   LoRA wins {lora_wins}/{len(metrics_order)} metrics")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Baseline T5 vs LoRA — Metrics Comparison", fontsize=15, fontweight="bold")
 
# Grouped bar
x = np.arange(len(metrics_order)); width = 0.35
b_vals = [baseline_metrics[m] for m in metrics_order]
l_vals = [lora_metrics[m]     for m in metrics_order]
axes[0].bar(x - width/2, b_vals, width, label="Baseline", color=PALETTE["baseline"])
axes[0].bar(x + width/2, l_vals, width, label="LoRA", color=PALETTE["lora"])
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics_order)
axes[0].set_ylim(0, 1.15); axes[0].set_title("Grouped Comparison"); axes[0].legend()
 
# Delta
deltas = [comparison_results[m]["delta"] for m in metrics_order]
axes[1].bar(metrics_order, deltas, color=PALETTE["lora"])
axes[1].axhline(0, color="black")
axes[1].set_title("LoRA Improvement (Δ)")
 
# Radar
angles = np.linspace(0, 2*np.pi, len(metrics_order), endpoint=False).tolist()
angles += angles[:1]
b_vals_r = b_vals + b_vals[:1]
l_vals_r = l_vals + l_vals[:1]
ax_radar = fig.add_subplot(133, polar=True)
ax_radar.plot(angles, b_vals_r, label="Baseline")
ax_radar.plot(angles, l_vals_r, label="LoRA")
ax_radar.set_thetagrids(np.degrees(angles[:-1]), metrics_order)
ax_radar.legend()
 
savefig("06b_metrics_comparison.png")

In [ ]:
print("\n📝 SAMPLE PREDICTIONS")
for i in range(min(5, len(eval_subset_base))):
    print("\nINPUT:", eval_subset_base[i]["input_text"])
    print("REF :", eval_subset_base[i]["target_text"])
    print("BASE:", baseline_preds[i])
    print("LORA:", lora_preds[i])

# Failure + Hallucination Detection

In [ ]:
failure_cases = []
hallucination_cases = []
 
for i, (pred, item) in enumerate(zip(baseline_preds, eval_subset_base)):
    ref = item["target_text"]
    rl  = rouge_metric.compute(predictions=[pred], references=[ref])["rougeL"]
 
    if rl < 0.35:
        failure_cases.append(i)
 
    nums_pred = set(re.findall(r"\d+", pred))
    nums_inp  = set(re.findall(r"\d+", item["input_text"]))
    if nums_pred - nums_inp:
        hallucination_cases.append(i)
 
print("Failures:", len(failure_cases))
print("Hallucinations:", len(hallucination_cases))

# Guardrails

In [ ]:
VALID_CLASSES   = set(CLASSES)
MAX_COUNT_VALUE = 50
 
def validate_counts(counts):
    return {k: min(max(int(v), 0), MAX_COUNT_VALUE) for k, v in counts.items()}
 
def validate_output(text):
    if len(text) < 10:
        return "[Invalid Output]"
    return text[:500]

# faiss setup


In [ ]:
import faiss
from transformers import AutoTokenizer, AutoModel
 
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embed_tok   = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(EMBED_MODEL).eval()

In [ ]:
def embed(text):
    enc = embed_tok(text, return_tensors="pt", truncation=True, max_length=64)
    with torch.no_grad():
        out = embed_model(**enc)
    return out.last_hidden_state.mean(dim=1).squeeze().numpy()

In [ ]:
faiss_index = faiss.IndexFlatL2(384)
faiss_meta = []
 
def store_vector(text):
    vec = embed(text).reshape(1, -1)
    faiss_index.add(vec)
    faiss_meta.append(text)
 
def retrieve_similar(query, k=3):
    vec = embed(query).reshape(1, -1)
    _, idxs = faiss_index.search(vec, k)
    return [faiss_meta[i] for i in idxs[0] if i < len(faiss_meta)]

In [ ]:
for i, img_path in enumerate(test_imgs[:20]):
    counts = run_yolo_on_image(str(img_path))
    score = compute_score(counts)
    prompt = build_lora_input(counts, score)
    expl = generate_text(t5_lora_model, prompt)
    store_vector(expl)

### retreival test

In [ ]:
query = "Highly cluttered urban street with many ads and vehicles"
results = retrieve_similar(query)
 
print("\n🔎 Retrieval Results:")
for r in results:
    print("-", r[:100])